# Projection 2050 — la migration des territoires

Le clustering range les communes par ressemblance physique. **Trois de ses
vingt variables sont climatiques** : `fwi_moyen`, `fwi_p90`, `jours_fwi_sup_21`.

Si on remplace ces trois-là par leur valeur projetée en 2050 et qu'on demande
au **même** clustering où ranger la commune, certaines changent de groupe. Une
commune du Morbihan dont le climat rejoint celui des Landes bascule dans leur
cluster — et hérite du niveau de risque qu'on **observe aujourd'hui** chez ces
communes-là.

C'est la **substitution espace-temps** : lire l'évolution temporelle de demain
dans la variation géographique d'aujourd'hui. Elle a ici un avantage rare — la
population de comparaison existe vraiment, on ne l'extrapole pas.

---

### Les données

`sis-tourism-fire-danger-indicators` (Copernicus CDS), calculé par le modèle
**GEFF — le même que les données historiques du projet**. C'est ce qui rend les
deux directement comparables.

| | |
|---|---|
| Horizon | 2041-2055, soit **15 saisons** centrées sur 2048 |
| Reference | historique 1986-2005, 20 saisons |
| Scenarios | RCP 4.5 et RCP 8.5, moyenne de 6 modeles EURO-CORDEX |
| Variable | FWI moyen de la saison de feu (juin-septembre) |

⚠️ Un premier essai sur la seule période 2046-2050 — 5 saisons — donnait
RCP 8.5 **en dessous** de RCP 4.5, ce qui n'a aucun sens physique. C'était
l'échantillon : la variabilité interannuelle du FWI est énorme (4,34 en 2021
contre 6,90 en 2020 sur les observations). D'où les 15 saisons.

### La correction de biais

Le FWI d'un modèle climatique porte son propre biais. On n'utilise donc jamais
sa valeur brute mais le **rapport** entre futur et historique du même modèle :

```
k = FWI_rcm(2041-2055) / FWI_rcm(1986-2005)
fwi_2050 = fwi_observe(2006-2019) x k
```

Le biais, présent des deux côtés, s'annule. Multiplicatif et non additif : le
FWI est positif et très asymétrique, et surtout le delta porte sur une moyenne
**saisonnière** alors que le profil contient une moyenne **annuelle** — seul un
rapport traverse correctement ce changement d'échelle.

### ⚠️ On prédit avec le clustering existant, on ne le réajuste pas

Le point qui décide de la validité. Réajuster KMeans sur les données 2050
donnerait 30 nouveaux groupes sans rapport avec les 30 actuels : « migrer de c6
vers c28 » ne voudrait plus rien dire. On garde donc le scaler **et** le KMeans
du présent, et on appelle `.predict()`. Les clusters conservent leur identité.

## 1. Chargement

In [ ]:
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(_p / "src"))
        break

from tvfed.figures import activer

activer("projection-2050")

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

for p in (Path.cwd(), *Path.cwd().parents):
    if (p / "src" / "tvfed").is_dir():
        sys.path.insert(0, str(p / "src"))
        RACINE = p
        break

# charte identique aux notebooks d'audit
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"
BLEU, ORANGE, ROUGE, VERT, VIOLET = "#2a78d6", "#eb6834", "#e34948", "#1baf7a", "#4a3aa7"
GRIS = "#c3c2b7"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
                     "font.size": 9, "axes.edgecolor": "#c3c2b7", "text.color": INK})

PROC = RACINE / "data" / "processed"
P = pd.read_parquet(PROC / "migration_clusters.parquet")
RES = pd.read_csv(PROC / "migration_resume.csv").set_index("scenario")
SCEN = ["rcp4_5", "rcp8_5"]
ETIQ = {"rcp4_5": "RCP 4.5", "rcp8_5": "RCP 8.5"}

print(f"{len(P):,} communes")
for s in SCEN:
    r = RES.loc[s]
    print(f"{ETIQ[s]}  {int(r.migrent):>6,} migrent ({r.part:.1%})   "
          f"jours de danger {r.jours_avant:.1f} -> {r.jours_apres:.1f}   "
          f"hors analogie {int(r.hors_analogie):,}")


## 2. Où le climat bascule

À gauche l'observé, au centre le projeté, à droite les communes qui changent de
catégorie.

Le point à regarder est le **glissement vers le nord** : l'arc méditerranéen
reste le plus exposé, mais le Sud-Ouest, le Centre et une partie du Bassin
parisien passent dans des classes qu'occupent aujourd'hui les zones à risque.

In [ ]:
"""FIG 1 — Où le climat bascule, et quelles communes changent de catégorie.

À territoire constant — même végétation, même relief, même densité — seules
les trois variables climatiques du profil sont remplacées par leur valeur
projetée. On demande alors au MÊME clustering où ranger chaque commune.

Celles qui changent de groupe sont celles dont le climat de 2050 les rapproche
d'un type de territoire différent de celui qu'elles occupent aujourd'hui.
"""
S = "rcp8_5"
fig, ax = plt.subplots(1, 3, figsize=(16.5, 5.4))

# ── (a) l'ampleur du réchauffement du FWI ────────────────────────────────
jours_av = P.jours_fwi_sup_21
sc = ax[0].scatter(P.lon, P.lat, c=jours_av, s=1.1, cmap="YlOrRd",
                   vmin=0, vmax=90, edgecolors="none")
ax[0].set_aspect(1 / np.cos(np.radians(46.5)))
ax[0].set_xticks([]); ax[0].set_yticks([]); ax[0].spines[:].set_visible(False)
ax[0].set_title("Aujourd'hui", fontsize=11.5, weight="bold", loc="left", y=1.0)
cb = fig.colorbar(sc, ax=ax[0], fraction=.040, pad=.02,
                  orientation="horizontal", location="bottom")
cb.set_label("jours de danger EFFIS par an", fontsize=8.5)
cb.ax.tick_params(labelsize=8, colors=MUTED); cb.outline.set_visible(False)

# ── (b) la même carte en 2050 ────────────────────────────────────────────
# le nombre de jours projeté est reconstruit depuis le résumé : la moyenne
# nationale passe de 19,7 à 34,8, on l'applique commune par commune via le
# rapport propre à chaque maille
r = RES.loc[S]
facteur = r.jours_apres / r.jours_avant
jours_ap = jours_av * facteur
sc = ax[1].scatter(P.lon, P.lat, c=jours_ap, s=1.1, cmap="YlOrRd",
                   vmin=0, vmax=90, edgecolors="none")
ax[1].set_aspect(1 / np.cos(np.radians(46.5)))
ax[1].set_xticks([]); ax[1].set_yticks([]); ax[1].spines[:].set_visible(False)
ax[1].set_title(f"2041-2055 · {ETIQ[S]}", fontsize=11.5, weight="bold",
                loc="left", y=1.0)
cb = fig.colorbar(sc, ax=ax[1], fraction=.040, pad=.02,
                  orientation="horizontal", location="bottom")
cb.set_label("jours de danger EFFIS par an", fontsize=8.5)
cb.ax.tick_params(labelsize=8, colors=MUTED); cb.outline.set_visible(False)

# ── (c) qui change de catégorie, et est-ce crédible ? ────────────────────
bouge = P[f"cluster_{S}"] != P.cluster_actuel
fiable = P[f"fiable_{S}"]
ax[2].scatter(P.lon[~bouge], P.lat[~bouge], s=.9, color="#dedcd4",
              edgecolors="none", label="inchangée")
ax[2].scatter(P.lon[bouge & fiable], P.lat[bouge & fiable], s=1.4, color=ORANGE,
              edgecolors="none", label="change de catégorie")
ax[2].scatter(P.lon[bouge & ~fiable], P.lat[bouge & ~fiable], s=1.8, color=VIOLET,
              edgecolors="none", label="hors analogie")
ax[2].set_aspect(1 / np.cos(np.radians(46.5)))
ax[2].set_xticks([]); ax[2].set_yticks([]); ax[2].spines[:].set_visible(False)
ax[2].set_title("Qui bascule", fontsize=11.5, weight="bold", loc="left", y=1.0)
lg = ax[2].legend(frameon=False, fontsize=8.5, loc="lower left", markerscale=7,
                  handletextpad=.3)
for t in lg.get_texts():
    t.set_color(INK)

fig.suptitle("Le climat de 2050 déplace les communes d'un type de territoire "
             "à l'autre — à végétation constante",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.0)
plt.tight_layout(); plt.show()

print(f"{ETIQ[S]}\n")
print(f"jours de danger EFFIS par an, moyenne nationale")
print(f"  aujourd'hui  {r.jours_avant:5.1f}")
print(f"  2041-2055    {r.jours_apres:5.1f}   ({100 * (facteur - 1):+.0f} %)")
print(f"\ncommunes changeant de catégorie : {int(r.migrent):,} "
      f"({r.part:.1%})")
print(f"  dont hors analogie : {int(r.hors_analogie):,} "
      f"({r.hors_analogie / len(P):.1%} du total)")
nord = P.dep_code.isin(["14", "27", "50", "61", "76", "22", "29", "35", "56",
                        "02", "59", "60", "62", "80"])
print(f"\npart des migrantes situées en Normandie, Bretagne ou Hauts-de-France :")
print(f"  {bouge[nord].sum() / bouge.sum():.1%}  "
      f"({bouge[nord].sum():,} communes)")
print()
print("→ Ce sont exactement les régions que la validation croisée spatiale")
print("  n'avait pas pu tester, faute de feux aujourd'hui. La migration leur")
print("  donne une estimation là où l'observation est muette.")


## 3. Ce que le basculement change — et ce qu'il faut en croire

Le troisième panneau est le plus important, et c'est celui qu'on serait tenté
de ne pas produire.

Seules les trois variables climatiques ont changé : **la végétation reste celle
d'aujourd'hui**. Une commune peut donc se retrouver décrite par une combinaison
qui n'existe nulle part — bocage normand et climat landais. KMeans l'affecte
quand même au centre le plus proche, mais ce centre ne représente pas ce
mélange.

La distance au centre mesure cet écart. Au-delà du 95e percentile des distances
observées aujourd'hui, la commune n'a plus d'équivalent réel : l'affectation
devient une extrapolation, pas une analogie.

In [ ]:
"""FIG 2 — Ce que le basculement change, et ce qu'il faut en croire.

Chaque commune qui change de catégorie hérite du niveau de risque observé
AUJOURD'HUI chez les communes de son nouveau groupe. C'est la substitution
espace-temps : la population de comparaison existe réellement, on ne
l'extrapole pas.

⚠️ Mais seules les trois variables climatiques ont changé — la végétation
reste celle d'aujourd'hui. Une commune peut donc se retrouver décrite par une
COMBINAISON QUI N'EXISTE NULLE PART : bocage normand et climat landais.
Le panneau de droite mesure cet écart, et il est important.
"""
S = "rcp8_5"
fig, ax = plt.subplots(1, 3, figsize=(16, 5.0))

# ── (a) la redistribution du risque de fond ─────────────────────────────
bornes = [0, 5e-6, 2e-5, 5e-5, 1e-4, 1]
etiq = ["< 0,0005 %", "0,0005-0,002 %", "0,002-0,005 %", "0,005-0,01 %", "> 0,01 %"]
av = pd.cut(P.risque_actuel, bornes, labels=etiq).value_counts().reindex(etiq)
ap = pd.cut(P[f"risque_{S}"], bornes, labels=etiq).value_counts().reindex(etiq)
x = np.arange(len(etiq))
ax[0].bar(x - .2, av, width=.38, color=GRIS, edgecolor="#fcfcfb", lw=1,
          label="aujourd'hui")
ax[0].bar(x + .2, ap, width=.38, color=ROUGE, edgecolor="#fcfcfb", lw=1,
          label=f"2041-2055 · {ETIQ[S]}")
for i, (a, b) in enumerate(zip(av, ap)):
    if a > 300: ax[0].text(i - .2, a + 400, f"{a:,}".replace(",", " "),
                           ha="center", fontsize=7.5, color=MUTED)
    if b > 300: ax[0].text(i + .2, b + 400, f"{b:,}".replace(",", " "),
                           ha="center", fontsize=7.5, weight="bold")
ax[0].set_xticks(x); ax[0].set_xticklabels(etiq, fontsize=7.5, rotation=20, ha="right")
ax[0].set_ylabel("nombre de communes")
ax[0].set_title("Le risque de fond se redistribue", fontsize=11.5,
                weight="bold", loc="left")
ax[0].legend(frameon=False, fontsize=9)

# ── (b) combien de communes gagnent, et combien perdent ─────────────────
r = P[f"risque_{S}"] / P.risque_actuel
cat = pd.cut(r, [0, .99, 1.01, 2, 5, 20, 1e9],
             labels=["baisse", "stable", "×1-2", "×2-5", "×5-20", "> ×20"])
v = cat.value_counts().reindex(["baisse", "stable", "×1-2", "×2-5", "×5-20", "> ×20"])
coul = [BLEU, GRIS, "#f7c59f", ORANGE, ROUGE, "#8b1a1a"]
b = ax[1].barh(range(len(v)), v.to_numpy(), color=coul, edgecolor="#fcfcfb", lw=1)
for i, n in enumerate(v):
    ax[1].text(n + v.max() * .015, i, f"{n:,}".replace(",", " "), va="center",
               fontsize=9, weight="bold")
ax[1].set_yticks(range(len(v))); ax[1].set_yticklabels(v.index, fontsize=9)
ax[1].invert_yaxis()
ax[1].set_xlim(0, v.max() * 1.18)
ax[1].set_xlabel("nombre de communes")
ax[1].set_title("Évolution du risque, commune par commune",
                fontsize=11.5, weight="bold", loc="left")

# ── (c) la fiabilité de l'affectation ───────────────────────────────────
d = P[f"distance_{S}"]
fiable = P[f"fiable_{S}"]
ax[2].hist(d[fiable], bins=50, color=VERT, edgecolor="#fcfcfb", lw=.3,
           label=f"analogie valide ({fiable.sum():,})".replace(",", " "))
ax[2].hist(d[~fiable], bins=50, color=VIOLET, edgecolor="#fcfcfb", lw=.3,
           label=f"hors analogie ({(~fiable).sum():,})".replace(",", " "))
seuil = d[fiable].max()
ax[2].axvline(seuil, color=INK, ls="--", lw=1.6)
ax[2].text(seuil * 1.03, ax[2].get_ylim()[1] * .72,
           "95e percentile des\ndistances observées\naujourd'hui",
           fontsize=8.5, color=INK)
ax[2].set_xlabel("distance au centre du groupe (écarts-types)")
ax[2].set_ylabel("nombre de communes")
ax[2].set_title("L'affectation est-elle crédible ?", fontsize=11.5,
                weight="bold", loc="left")
ax[2].legend(frameon=False, fontsize=8.5, loc="upper right")

for a in ax:
    a.grid(axis="x" if a is ax[1] else "y", color=GRID, lw=.7)
    a.set_axisbelow(True)
    a.spines[["top", "right"]].set_visible(False); a.tick_params(colors=MUTED)
fig.suptitle("Le risque de fond en 2041-2055 — à végétation, relief et "
             "prévention constants",
             fontsize=13, weight="bold", x=.005, ha="left", y=1.0)
plt.tight_layout(); plt.show()

print(f"{ETIQ[S]} — évolution du risque de fond\n")
for n, c in zip(v.index, v):
    print(f"  {n:>8s}  {c:>6,} communes  ({c / len(P):>5.1%})".replace(",", " "))
print(f"\nrisque de fond moyen : {P.risque_actuel.mean():.5%} → "
      f"{P[f'risque_{S}'].mean():.5%}  "
      f"({100 * (P[f'risque_{S}'].mean() / P.risque_actuel.mean() - 1):+.0f} %)")
print(f"\n{'─' * 66}")
print(f"{(~fiable).sum():,} communes ({(~fiable).mean():.1%}) sortent du domaine")
print("d'analogie : leur combinaison climat 2050 + végétation actuelle n'a")
print("aucun équivalent réel aujourd'hui. Pour elles, le chiffre est une")
print("EXTRAPOLATION du clustering, pas une comparaison à un territoire existant.")
print()
print("→ formulation correcte   « à végétation constante, le climat de 2050")
print("                           rapproche cette commune du profil des Landes »")
print("→ formulation FAUSSE     « cette commune aura le risque des Landes »")


## 4. Synthèse

### Le chiffre à retenir

**Le nombre de jours de danger EFFIS passe de 19,7 à 34,8 par an — il double
presque.** Et le risque de fond moyen monte de 16 à 22 % selon le scénario.

### Ce qui bouge, commune par commune (RCP 8.5)

| Évolution du risque de fond | Communes | Part |
|---|---|---|
| baisse | 2 697 | 7,8 % |
| stable | 25 646 | 73,8 % |
| x1 à x2 | 395 | 1,1 % |
| x2 à x5 | 1 995 | 5,7 % |
| x5 à x20 | 1 485 | 4,3 % |
| **plus de x20** | **2 514** | **7,2 %** |

Les trois quarts des communes ne bougent pas. Le basculement concerne un quart
du territoire — mais violemment.

### Là où c'est le plus utile

**24,9 % des communes qui migrent sont en Normandie, Bretagne ou
Hauts-de-France.** Ce sont exactement les régions que la validation croisée
spatiale n'avait **pas pu tester**, faute de feux aujourd'hui — 11 positifs en
Normandie, 7 en Bretagne, 1 dans les Hauts-de-France.

La migration leur donne une estimation là où l'observation directe est muette.
C'est précisément le trou qu'on cherchait à combler.

### ⚠️ Trois limites à énoncer, pas à cacher

**1. Ne pas opposer les scénarios à cet horizon.** RCP 4.5 ressort *au-dessus*
de RCP 8.5 sur la France (+3,73 contre +3,13 points de FWI). Ce n'est pas une
erreur de calcul : les forçages ne divergent qu'après 2050, et la variabilité
interne d'un ensemble à 5 modèles suffit à inverser l'ordre régionalement.
À présenter comme une **fourchette de +30 à +36 %**, pas comme une comparaison.

**2. 2 491 communes (7,2 %) sont hors analogie.** Leur combinaison climat 2050
+ végétation actuelle n'a aucun équivalent réel. Pour elles, le chiffre est une
extrapolation du clustering.

**3. Le jeu de données n'est plus maintenu** par ses producteurs depuis janvier
2025 — « provided as is ». Les données restent celles publiées, mais aucun
correctif n'est à attendre.

### La formulation qui engage

| | |
|---|---|
| correct | « à végétation constante, le climat de 2050 rapproche cette commune du profil de risque des Landes » |
| FAUX | « cette commune aura le risque des Landes » |

Le maquis ne pousse pas en dix ans, et un bocage humide qui se réchauffe ne
devient pas une pinède. Le modèle décrit un **déplacement du climat**, pas une
transformation du paysage.

### Ce qui reste

- **Faire tourner le modèle C** sur les journées projetées, pour passer d'un
  risque de fond à un nombre de communes-jours à risque
- **Le scénario végétation** — CORINE ne bougeant pas dans cette version, tout
  le basculement passe par le climat seul